# Create Daily Web Publication

## Load useful libraries

In [ ]:
import os
import json
import shutil
from pathlib import Path

In [ ]:
from strategic_reports.daily.inators.report_inator import produce_article_summary_markdown_report
from strategic_reports.daily.inators.report_inator import produce_strategic_markdown_report
from strategic_reports.daily.inators.report_inator import convert_markdown_to_html
from strategic_reports.daily.inators.report_inator import write_html
from strategic_reports.daily.inators.html_fix_inator import html_fix_assemble

In [ ]:
from strategic_reports.daily.config.topic_order import list_directories_and_titles

## User settings

In [ ]:
directory_output_root = os.environ['STRATEGIC_REPORTS_HOME'] + '/output/daily'
subdirectory_name_report = 'strategic-report'
subdirectory_name_logs = 'logs'
report_header_level = '#'
report_title = 'Emily\'s Strategic Review'
ai_content_per_topic = 'autonomous'
ai_content_all_topics = 'autonomous'
debug = False
debug_group = None

## Set debug parameters

In [ ]:
if debug:
    list_directories_and_titles = [x for x in list_directories_and_titles if x['slug'] in debug_group]

In [ ]:
if debug:
    print(list_directories_and_titles)

## Clear and then create a path for output

In [ ]:
shutil.rmtree(directory_output_root + '/' + subdirectory_name_report, ignore_errors = True)
Path(directory_output_root + '/' + subdirectory_name_report).mkdir()

## Figure out which topics failed to produce content

In [ ]:
with open(directory_output_root + '/' + subdirectory_name_logs + '/pipeline_failures.json') as f:
    list_failed_reports = json.load(f)['pipeline_failures']

In [ ]:
list_directories_and_titles_to_use = [x for x in list_directories_and_titles if not x['slug'] in list_failed_reports]

## Ensure there is at lease one successful topic before continuing

In [ ]:
assert len(list_directories_and_titles_to_use) >= 1

## Generate the summaries document for each topic

In [ ]:
for item in list_directories_and_titles_to_use:
    subdirectory = item['slug']
    title = item['title']

    # maybe this filename should be a variable
    with open(directory_output_root + '/' + subdirectory + '/summaries_and_llm_tags.json') as f:
        dict_summaries_tags = json.load(f)

    summaries_report = produce_article_summary_markdown_report(dict_summaries_tags, title = title)
    
    html = convert_markdown_to_html(summaries_report)
    html = html_fix_assemble(html, ai_content = ai_content_per_topic)
    
    write_html(html, directory_output_root + '/' + subdirectory_name_report + '/' + subdirectory.replace('feeds_', '') + '_summaries.html')  

## Generate and save the strategic summaries

In [ ]:
if len(list_directories_and_titles_to_use) > 0:
    report = produce_strategic_markdown_report(
        list_directories_and_titles_to_use,
        directory_output_root,
        report_header_level = report_header_level,
    )

In [ ]:
with open(directory_output_root + '/' + subdirectory_name_report + '/index.md', 'w') as f:
    f.write(report + '\n')

In [ ]:
html = convert_markdown_to_html(report)

In [ ]:
html = html_fix_assemble(html, ai_content = ai_content_all_topics, title = report_title)

In [ ]:
write_html(html, directory_output_root + '/' + subdirectory_name_report + '/index.html')